## Models prediction

In [2]:
import pandas as pd
file_path ='/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/data/processed/divemt_model_input.tsv'
model_df_processed = pd.read_csv(file_path, sep ='\t')

### 1. Create train-test set using GroupKFold cross validation

In [3]:
import sklearn as sk
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

In [4]:
language_splits = {}

for lang in model_df_processed['lang_id'].unique():
    lang_df = model_df_processed.loc[model_df_processed['lang_id']==lang].copy()
    doc_id = lang_df['doc_id']
    gkf = GroupKFold (n_splits=10)
    splits = list(gkf.split(lang_df, groups = doc_id))
    language_splits[lang] = splits

### 2. Create prediction function

In [5]:
def run_cv_model(lang_df, splits, feature_cols):
    
    fold_results = []
    prediction_results =[]

    X = lang_df[feature_cols]
    y = lang_df['edit_time']

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model = LinearRegression()
        model.fit(X_train,y_train)

        y_pred=model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rho, p_value = spearmanr(y_test, y_pred)

        fold_results.append({
            "fold": fold,
            "mae": mae,
            "spearman_rho": rho,
            "spearman_p": p_value
            })
        # Raw prediction for every test observation
        test_rows = lang_df.iloc[test_idx]

        for unit_id, doc_id, actual, predicted in zip(
            test_rows["unit_id"],
            test_rows["doc_id"],
            y_test,
            y_pred
        ):
            prediction_results.append({
                "unit_id": unit_id,
                "doc_id": doc_id,
                "fold": fold,
                "actual_edit_time": actual,
                "predicted_edit_time": predicted
            })

    return (
        pd.DataFrame(fold_results),
        pd.DataFrame(prediction_results)
    )
        

In [6]:
#Define features for each prediction model
feature_sets = {
    "baseline": [
        "mt_len_wrd",
        "mt_system"
    ],
    "cometkiwi": [
        "mt_len_wrd",
        "mt_system",
        "qe_cometkiwi"
    ],
    "gemba_gemma": [
        "mt_len_wrd",
        "mt_system",
        "qe_gemba_gemma3"
    ],
    "gemba_gpt": [
        "mt_len_wrd",
        "mt_system",
        "qe_gemba_gpt4_1_mini"
    ]
}

### 3. Create loop to run prediction across 6 languages & 4 models

In [8]:
all_results = []
all_predictions =[]
for lang in model_df_processed['lang_id'].unique():
    lang_df = model_df_processed.loc[model_df_processed['lang_id']==lang].copy()
    fold = language_splits [lang]

    for model, features in feature_sets.items():
        fold_result, predictions = run_cv_model (lang_df = lang_df, splits = fold, feature_cols = features)

        fold_result['lang_id'] =lang
        fold_result['model'] = model

        predictions["lang_id"] = lang
        predictions["model"] = model

        all_results.append(fold_result)
        all_predictions.append(predictions)


results_df = pd.concat(
    all_results,
    ignore_index=True
)

predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)


In [9]:
"""
results_df.to_csv(
    "../results/linear_regression_cv_fold_results.tsv",
    sep="\t",
    index=False
)
"""

predictions_df.to_csv(
    "../results/linear_regression_cv_predictions.tsv",
    sep="\t",
    index=False
)

print(results_df.shape)
print(results_df.head(10))

(240, 6)
   fold        mae  spearman_rho    spearman_p lang_id     model
0     1  42.579809      0.394785  1.410707e-04     tur  baseline
1     2  36.464302      0.366984  4.365540e-04     tur  baseline
2     3  26.917462      0.617867  1.427666e-10     tur  baseline
3     4  46.569510      0.396029  1.337955e-04     tur  baseline
4     5  34.712020      0.458154  7.172640e-06     tur  baseline
5     6  51.284120      0.509935  3.881135e-07     tur  baseline
6     7  36.809080      0.470533  8.166044e-06     tur  baseline
7     8  33.808672      0.453706  1.861568e-05     tur  baseline
8     9  55.971701      0.450960  2.120883e-05     tur  baseline
9    10  37.329330      0.533916  1.197621e-07     tur  baseline


In [10]:
test = pd.read_csv('/Users/tuedungtruong/Documents/PythonProjects/msc-thesis/results/linear_regression_cv_fold_results.tsv', sep ='\t')
test.head(10)

,fold,mae,spearman_rho,spearman_p,lang_id,model
0,1,42.579809,0.394785,1.410707e-04,tur,baseline
1,2,36.464302,0.366984,4.365540e-04,tur,baseline
2,3,26.917462,0.617867,1.427666e-10,tur,baseline
3,4,46.569510,0.396029,1.337955e-04,tur,baseline
4,5,34.712020,0.458154,7.172640e-06,tur,baseline
5,6,51.284120,0.509935,3.881135e-07,tur,baseline
6,7,36.809080,0.470533,8.166044e-06,tur,baseline
7,8,33.808672,0.453706,1.861568e-05,tur,baseline
8,9,55.971701,0.450960,2.120883e-05,tur,baseline
9,10,37.329330,0.533916,1.197621e-07,tur,baseline


In [11]:
test.groupby(['lang_id','model']).describe()

fold                                             mae  \
                    count mean      std  min   25%  50%   75%   max count   
lang_id model                                                               
ara     baseline     10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        cometkiwi    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gemma  10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gpt    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
ita     baseline     10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        cometkiwi    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gemma  10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gpt    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
nld     baseline     10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        cometkiwi    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gemma  10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gpt    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
tur     baseline     10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        cometkiwi    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gemma  10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gpt    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
ukr     baseline     10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        cometkiwi    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gemma  10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gpt    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
vie     baseline     10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        cometkiwi    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gemma  10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   
        gemba_gpt    10.0  5.5  3.02765  1.0  3.25  5.5  7.75  10.0  10.0   

                                ... spearman_rho           spearman_p  \
                          mean  ...          75%       max      count   
lang_id model                   ...                                     
ara     baseline     80.833186  ...     0.588706  0.649166       10.0   
        cometkiwi    78.041535  ...     0.598233  0.707037       10.0   
        gemba_gemma  79.704927  ...     0.541009  0.627172       10.0   
        gemba_gpt    78.921849  ...     0.589747  0.629408       10.0   
ita     baseline     56.444111  ...     0.460399  0.498459       10.0   
        cometkiwi    53.602100  ...     0.525489  0.623155       10.0   
        gemba_gemma  55.484336  ...     0.509783  0.557095       10.0   
        gemba_gpt    54.149555  ...     0.529178  0.629629       10.0   
nld     baseline     42.006722  ...     0.413595  0.500487       10.0   
        cometkiwi    41.402634  ...     0.480160  0.550861       10.0   
        gemba_gemma  41.800635  ...     0.435811  0.527185       10.0   
        gemba_gpt    41.323663  ...     0.480088  0.516183       10.0   
tur     baseline     40.244601  ...     0.500084  0.617867       10.0   
        cometkiwi    39.311082  ...     0.535870  0.658924       10.0   
        gemba_gemma  39.710437  ...     0.520796  0.625106       10.0   
        gemba_gpt    39.429937  ...     0.523818  0.652873       10.0   
ukr     baseline     73.639011  ...     0.581732  0.647128       10.0   
        cometkiwi    72.595156  ...     0.609348  0.666814       10.0   
        gemba_gemma  72.840240  ...     0.588620  0.663927       10.0   
        gemba_gpt    72.914156  ...     0.590193  0.666887       10.0   
vie     baseline     85.248737  ...     0.466169  0.595938       10.0   
        cometkiwi    81.881375  ...     0.563170  0.680433       10.0   
        gemba_gemma  84.389353  ...     0.518102  0.654993       10.0   
        gemba_gpt    83.869835  ...     0.5

In [12]:
model_df_processed.groupby('lang_id')['lang_id'].describe()

,count,unique,top,freq
lang_id,,,,
ara,858,1,ara,858
ita,860,1,ita,860
nld,860,1,nld,860
tur,860,1,tur,860
ukr,860,1,ukr,860
vie,849,1,vie,849
